## SwasthyaSetu Final Pages
After OCR, every one of the 174 pages gets one decision: does it go into search or not. One diagram gets rewritten by hand, every page becomes one clean record, and pages.jsonl is saved only when all tests pass.

## 174 pages in → 114 searchable 60 stored only → 20 tests → 1 file: pages.jsonl

## 16.0 Setup


In [1]:
import copy, json
from collections import Counter
import pandas as pd
from src.paths import PAGES_OCR, LETTERS_JSONL, PAGE_MAP_CSV, PAGE_TYPES_CSV, PAGES_FINAL

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]

PAGES   = read_jsonl(PAGES_OCR)
LETTERS = read_jsonl(LETTERS_JSONL)

_types    = pd.read_csv(PAGE_TYPES_CSV)                       # row order = PDF page order
PAGE_TYPE = dict(zip(range(1, len(_types) + 1), _types["label"]))

_pm         = pd.read_csv(PAGE_MAP_CSV, dtype=str)
FOOTER_SEEN = dict(zip(_pm["pdf_page"].astype(int), _pm["status"] == "Matched"))

LETTER_OF = {pg: l["letter_no"] for l in LETTERS for pg in l["pdf_pages"]}
print(f"{len(PAGES)} pages · {len(LETTERS)} letters covering {len(LETTER_OF)} pages")

174 pages · 21 letters covering 37 pages


## 16.1 Page policy (registry: the first rule that matches wins)

In [2]:
NO_CONTENT = {"cover", "title / other", "back cover", "notes", "blank"}
CONTENT    = {"chapter body", "annexure I (procedures)", "annexure form", "case study", "abbreviations"}

PAGE_RULES = [
    {"reason": "blank page",                "search": False, "when": lambda p: p.get("status") == "blank"},
    {"reason": "part of a letter",          "search": False, "when": lambda p: p["pdf_page"] in LETTER_OF},
    {"reason": "no content",                "search": False, "when": lambda p: p["page_type"] in NO_CONTENT},
    {"reason": "contents (for step 20)",    "search": False, "when": lambda p: p["page_type"] == "contents"},
    {"reason": "message (v1: stored only)", "search": False, "when": lambda p: p["page_type"] == "scanned message"},
    {"reason": "unreadable scan",           "search": False, "when": lambda p: p.get("status") == "low_quality"},
    {"reason": "content",                   "search": True,  "when": lambda p: p["page_type"] in CONTENT},
]

def decide(page):
    for rule in PAGE_RULES:
        if rule["when"](page):
            return rule["search"], rule["reason"]
    raise ValueError(f"PDF {page['pdf_page']}: no rule for page type {page['page_type']!r}")

def with_type(p):
    return {**p, "page_type": PAGE_TYPE[p["pdf_page"]]}       # new dict, PAGES untouched

DECISIONS = pd.DataFrame([{"pdf_page": p["pdf_page"], "page_type": PAGE_TYPE[p["pdf_page"]],
                           **dict(zip(("search", "reason"), decide(with_type(p))))} for p in PAGES])
DECISIONS.groupby(["reason", "search"]).size()

reason                     search
blank page                 False      12
content                    True      114
contents (for step 20)     False       4
message (v1: stored only)  False       5
no content                 False       2
part of a letter           False      37
dtype: int64

## 17 PDF 32 diagram → manual text


In [3]:
DIAGRAMS = {
    32: {"caption": "Exhibit 1: Key steps in the Process of Recruitment",
         "steps": ["Identify and validate vacancies.",
                   "Review Job Description.",
                   "Design and disseminate advertisement for vacancy.",
                   "Decide on stages of selection and special weightage criteria, if any.",
                   "Review CVs and shortlist applications.",
                   "Proceed through all selection stages.",
                   "Reference check of selected candidates.",
                   "Disseminate result and issue offer letter."]},
}

def replace_diagram(lines, spec):
    """Replace the body lines after the caption (up to the next non-body line) with numbered steps.
    Returns (new_lines, removed_lines). The input list is not changed."""
    start = next(i for i, l in enumerate(lines)
                 if l["level"] == "caption" and l["text"].startswith(spec["caption"]))
    end = next((i for i in range(start + 1, len(lines)) if lines[i]["level"] != "body"), len(lines))
    manual = [{"text": f"{n}. {s}", "level": "diagram", "source": "manual"}
              for n, s in enumerate(spec["steps"], 1)]
    return lines[:start + 1] + manual + lines[end:], lines[start + 1:end]

_, removed = replace_diagram(next(p for p in PAGES if p["pdf_page"] == 32)["lines"], DIAGRAMS[32])
[l["text"] for l in removed]          # look: should be the 12 jumbled box lines

['Design and Disseminate',
 'Identify and validate Review Job',
 'Advertisement for',
 'vacancies. Description',
 'vacancy',
 'Decide on stages of',
 'Proceed through all Review CVs and',
 'selection and special',
 'selection stages. shortlist applications',
 'weightage criteria, if any.',
 'Reference check of Disseminate result and',
 'selected candidates issue offerletter.']

## 18 Build final records


In [4]:
def label_source(pdf_no):
    if pdf_no in (1, 2):
        return "override"
    return "footer" if FOOTER_SEEN[pdf_no] else "pdf_label"

def build_page(p):
    p = copy.deepcopy(with_type(p))
    no = p["pdf_page"]
    search, reason = decide(p)
    lines = replace_diagram(p["lines"], DIAGRAMS[no])[0] if no in DIAGRAMS else p["lines"]
    return {"pdf_page": no, "printed_page": p["printed_page"], "label_source": label_source(no),
            "part": p["part"], "page_type": p["page_type"], "method": p["method"],
            "status": p.get("status") or "ok", "search": search, "reason": reason,
            "letter_no": LETTER_OF.get(no), "lines": lines, "footnotes": p["footnotes"],
            "tables": p["tables"],
            "notice": p.get("notice") if reason == "unreadable scan" else None,
            "manual": no in DIAGRAMS}

FINAL = [build_page(p) for p in PAGES]
FINAL_BY_NO = {r["pdf_page"]: r for r in FINAL}
print(f"{len(FINAL)} records · {sum(r['search'] for r in FINAL)} searchable")

174 records · 114 searchable


## 19 Tests (use the same check() as notebook 03, pasted first)


In [7]:
TESTS = []
def check(group, name, get_actual, expected, desc=None):
    try:
        actual = get_actual()
        ok = expected(actual) if callable(expected) else actual == expected
    except Exception as e:
        actual, ok = f"ERROR {type(e).__name__}: {e}", False
    TESTS.append({"group": group, "test": name, "result": "PASS" if ok else "FAIL",
                  "expected": desc if callable(expected) else expected, "actual": actual})


def raises(fn):
    """check() treats an exception as FAIL, so to test that something SHOULD fail, return the error name."""
    try:
        fn(); return "no error"
    except Exception as e:
        return type(e).__name__

def line_after_diagram(rec):
    levels = [l["level"] for l in rec["lines"]]
    last = len(levels) - 1 - levels[::-1].index("diagram")
    return rec["lines"][last + 1]["text"]

G = "16 include"
check(G, "page types for all 174 pages", lambda: sorted(PAGE_TYPE) == list(range(1, 175)), True)
check(G, "letters cover PDF 109-145 exactly once",
      lambda: (sorted(LETTER_OF), sum(len(l["pdf_pages"]) for l in LETTERS)), (list(range(109, 146)), 37))
check(G, "pages per reason", lambda: dict(Counter(r["reason"] for r in FINAL)),
      {"blank page": 12, "part of a letter": 37, "no content": 2, "contents (for step 20)": 4,
       "message (v1: stored only)": 5, "content": 114})
check(G, "searchable = content types only",
      lambda: sorted({r["page_type"] for r in FINAL if r["search"]}), sorted(CONTENT))
check(G, "no letter page searchable", lambda: [r["pdf_page"] for r in FINAL if r["letter_no"] and r["search"]], [])
check(G, "messages kept with text, not searchable",
      lambda: [(r["pdf_page"], r["search"], bool(r["lines"])) for r in FINAL if r["page_type"] == "scanned message"],
      [(n, False, True) for n in (5, 7, 9, 11, 13)])
check(G, "every searchable page has text or tables",
      lambda: [r["pdf_page"] for r in FINAL if r["search"] and not (r["lines"] or r["tables"])], [])
check(G, "no blank/low-quality page searchable",
      lambda: [r["pdf_page"] for r in FINAL if r["search"] and r["status"] != "ok"], [])

G = "17 diagram"
check(G, "PDF 32: 8 manual steps in order",
      lambda: [l["text"] for l in FINAL_BY_NO[32]["lines"] if l["level"] == "diagram"],
      [f"{n}. {s}" for n, s in enumerate(DIAGRAMS[32]["steps"], 1)])
check(G, "PDF 32: jumbled box text gone",
      lambda: [l["text"] for l in FINAL_BY_NO[32]["lines"]
               if l["text"] in ("vacancy", "selection stages. shortlist applications")], [])
check(G, "PDF 32: heading 5.3.1 right after diagram",
      lambda: line_after_diagram(FINAL_BY_NO[32]), "5.3.1. Identification and Validation of Vacancies")
check(G, "PAGES not changed by build",
      lambda: any(l["level"] == "diagram" for l in next(p for p in PAGES if p["pdf_page"] == 32)["lines"]), False)
check(G, "manual flag only on diagram pages", lambda: [r["pdf_page"] for r in FINAL if r["manual"]], sorted(DIAGRAMS))

G = "18 final"
check(G, "174 records, PDF 1..174 in order", lambda: [r["pdf_page"] for r in FINAL] == list(range(1, 175)), True)
check(G, "same keys on every record", lambda: len({tuple(r) for r in FINAL}), 1)
check(G, "printed = PDF - 20 (PDF 21-174)",
      lambda: [r["pdf_page"] for r in FINAL if r["pdf_page"] > 20 and r["printed_page"] != str(r["pdf_page"] - 20)], [])
check(G, "label_source counts", lambda: dict(Counter(r["label_source"] for r in FINAL)),
      {"override": 2, "pdf_label": 17, "footer": 155})
check(G, "notice only on unreadable non-letter pages",
      lambda: [r["pdf_page"] for r in FINAL if r["notice"] and r["reason"] != "unreadable scan"], [])

G = "16-18 unit"
check(G, "unknown page type fails loudly",
      lambda: raises(lambda: decide({"pdf_page": 999, "page_type": "poster", "status": "ok"})), "ValueError")
_fake = [{"text": "Exhibit 9: X", "level": "caption"}, {"text": "box b", "level": "body"},
         {"text": "box a", "level": "body"}, {"text": "1.1 Next", "level": "L2"}]
check(G, "replace_diagram swaps only up to next heading",
      lambda: [l["text"] for l in replace_diagram(_fake, {"caption": "Exhibit 9", "steps": ["a", "b"]})[0]],
      ["Exhibit 9: X", "1. a", "2. b", "1.1 Next"])
# + the same report / n_pass lines as notebook 03


report = pd.DataFrame(TESTS)
n_pass = int((report["result"] == "PASS").sum())
print(f"{'✅' if n_pass == len(report) else '❌'}  {n_pass}/{len(report)} tests passed")
failed = report[report["result"] == "FAIL"]
failed if len(failed) else "All tests passed — nothing to show."

✅  20/20 tests passed


'All tests passed — nothing to show.'

In [8]:
if n_pass == len(report):
    with PAGES_FINAL.open("w", encoding="utf-8") as f:
        for r in FINAL:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    assert read_jsonl(PAGES_FINAL) == FINAL, "reload differs from what was saved"
    print(f"saved {len(FINAL)} pages → {PAGES_FINAL.name} (reload identical)")
else:
    print("NOT saved — fix the failing tests first")

saved 174 pages → pages.jsonl (reload identical)
